# Snowflake MCP Server Setup (Trial)

This notebook is designed to provision your Snowflake MCP-related objects in a trial environment using values from your workspace .env file.

Important:
- This notebook does not print secrets.
- This notebook can auto-generate a baseline CREATE MCP SERVER statement.
- You can still provide your own provisioning SQL via environment variable.
- You can optionally provision a Cortex Search Service before MCP server creation.

## 1) Install Python dependencies
Run this once in the active kernel.

In [ ]:
%pip install -q snowflake-connector-python python-dotenv

## 2) Load environment and validate required values
This cell loads the repository .env and verifies connection settings.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv

start_dir = Path.cwd().resolve()
env_path = None
for candidate_dir in [start_dir, *start_dir.parents]:
    candidate_env = candidate_dir / '.env'
    if candidate_env.exists():
        env_path = candidate_env
        break

if env_path is None:
    searched = [str(p / '.env') for p in [start_dir, *start_dir.parents]]
    raise FileNotFoundError('Missing .env file. Checked: ' + '; '.join(searched))

repo_root = env_path.parent

# Always refresh notebook env values from file in the current kernel session.
load_dotenv(env_path, override=True)

required = [
    'SNOWFLAKE_ACCOUNT',
    'SNOWFLAKE_ADMIN_USER',
    'SNOWFLAKE_ADMIN_PASSWORD',
    'SNOWFLAKE_ADMIN_ROLE',
    'SNOWFLAKE_WAREHOUSE',
]

missing = [key for key in required if not os.getenv(key)]
if missing:
    raise ValueError('Missing required .env values: ' + ', '.join(missing))

connection_config = {
    'account': os.getenv('SNOWFLAKE_ACCOUNT'),
    'user': os.getenv('SNOWFLAKE_ADMIN_USER'),
    'password': os.getenv('SNOWFLAKE_ADMIN_PASSWORD'),
    'role': os.getenv('SNOWFLAKE_ADMIN_ROLE'),
    'warehouse': os.getenv('SNOWFLAKE_WAREHOUSE'),
    'database': os.getenv('SNOWFLAKE_DATABASE') or None,
    'schema': os.getenv('SNOWFLAKE_SCHEMA') or None,
}

safe_preview = {
    'account': connection_config['account'],
    'user': connection_config['user'],
    'role': connection_config['role'],
    'warehouse': connection_config['warehouse'],
    'database': connection_config['database'],
    'schema': connection_config['schema'],
}

verbose = os.getenv('SNOWFLAKE_MCP_VERBOSE', 'false').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

print('Environment loaded. Ready for Snowflake preflight and MCP provisioning.')
print('Target context:', f"{safe_preview['database']}.{safe_preview['schema']}")


## 3) Connect and run a Snowflake preflight check

In [ ]:
import snowflake.connector
from snowflake.connector import DictCursor

conn = snowflake.connector.connect(**connection_config)

with conn.cursor(DictCursor) as cur:
    cur.execute(
        """
        SELECT
            CURRENT_VERSION() AS current_version,
            CURRENT_ACCOUNT() AS current_account,
            CURRENT_ROLE() AS current_role,
            CURRENT_WAREHOUSE() AS current_warehouse
        """
    )
    row = cur.fetchone()

status = {k.lower(): v for k, v in row.items()}

print('Connected successfully.')
print('Version:', status['current_version'])
print('Account:', status['current_account'])
print('Role:', status['current_role'])
print('Warehouse:', status['current_warehouse'])

## 4) Optional: Provision NL2SQL and retrieval dependencies
This step auto-provisions the Semantic View (required for Cortex Analyst NL2SQL) and optionally a Cortex Search Service.

**Semantic View provisioning (.env):**
- SNOWFLAKE_MCP_PROVISION_SEMANTIC_VIEW=true -- enables this step
- SNOWFLAKE_MCP_SEMANTIC_VIEW_TEMPLATE=tpch -- use the built-in TPCH template (orders, customers, lineitems, nations, regions)
- SNOWFLAKE_MCP_SEMANTIC_VIEW_DATABASE=<target db for the semantic view>
- SNOWFLAKE_MCP_SEMANTIC_VIEW_SCHEMA=<target schema>
- SNOWFLAKE_MCP_SEMANTIC_VIEW_NAME=<name>
- SNOWFLAKE_MCP_SEMANTIC_VIEW_SQL=<raw CREATE SEMANTIC VIEW SQL> (overrides template)
- SNOWFLAKE_MCP_SV_SOURCE_DATABASE=<source db, default SNOWFLAKE_SAMPLE_DATA> (for tpch template)
- SNOWFLAKE_MCP_SV_SOURCE_SCHEMA=<source schema, default TPCH_SF1> (for tpch template)

This step creates the target schema if it does not exist, then creates the semantic view.
On success it sets SNOWFLAKE_MCP_ANALYST_IDENTIFIER in the kernel session for Step 5 to use.

**Optional Cortex Search provisioning (.env):**
- SNOWFLAKE_MCP_PROVISION_CORTEX_SEARCH=true
- SNOWFLAKE_MCP_SEARCH_DATABASE, SNOWFLAKE_MCP_SEARCH_SCHEMA, SNOWFLAKE_MCP_SEARCH_SERVICE_NAME
- SNOWFLAKE_MCP_SEARCH_ON_COLUMN, SNOWFLAKE_MCP_SEARCH_SOURCE_SQL
- SNOWFLAKE_MCP_SEARCH_WAREHOUSE, SNOWFLAKE_MCP_SEARCH_ATTRIBUTES, SNOWFLAKE_MCP_SEARCH_TARGET_LAG

In [ ]:
import re
from snowflake.connector import DictCursor

def _validate_ident(value: str, name: str) -> str:
    if not value or not value.strip():
        raise ValueError(f'{name} is required.')
    v = value.strip()
    if not re.match(r'^[A-Za-z_][A-Za-z0-9_$]*$', v):
        raise ValueError(f'{name} has unsupported characters: {value}')
    return v

def _as_bool_local(value: str | None, default: bool = False) -> bool:
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

def _parse_three_part_identifier(identifier: str, env_name: str) -> tuple[str, str, str]:
    parts = [p.strip() for p in identifier.split('.') if p.strip()]
    if len(parts) != 3:
        raise ValueError(f'{env_name} must be in the format database.schema.object_name.')
    return parts[0], parts[1], parts[2]

def _collect_semantic_views(conn_obj, database: str | None = None, schema: str | None = None):
    with conn_obj.cursor(DictCursor) as cur:
        if database and schema:
            cur.execute(f'SHOW SEMANTIC VIEWS IN SCHEMA {database}.{schema}')
        else:
            cur.execute('SHOW SEMANTIC VIEWS IN ACCOUNT')
        rows = cur.fetchall()
    results = []
    for row in rows:
        normalized = {str(k).lower(): v for k, v in row.items()}
        name = str(normalized.get('name') or '').strip()
        database_name = str(normalized.get('database_name') or normalized.get('database') or '').strip()
        schema_name = str(normalized.get('schema_name') or normalized.get('schema') or '').strip()
        if name and database_name and schema_name:
            results.append((database_name, schema_name, name))
    return results

def _match_semantic_views(views, db, schema, name):
    db_u = db.upper() if db else None
    schema_u = schema.upper() if schema else None
    name_u = name.upper() if name else None
    matches = []
    for view_db, view_schema, view_name in views:
        if db_u and view_db.upper() != db_u:
            continue
        if schema_u and view_schema.upper() != schema_u:
            continue
        if name_u and view_name.upper() != name_u:
            continue
        matches.append((view_db, view_schema, view_name))
    return matches

def _build_tpch_semantic_view_sql(target_db: str, target_schema: str, view_name: str) -> str:
    src_db = (os.getenv('SNOWFLAKE_MCP_SV_SOURCE_DATABASE') or 'SNOWFLAKE_SAMPLE_DATA').strip()
    src_schema = (os.getenv('SNOWFLAKE_MCP_SV_SOURCE_SCHEMA') or 'TPCH_SF1').strip()
    return f'''CREATE OR REPLACE SEMANTIC VIEW {target_db}.{target_schema}.{view_name}
  TABLES (
    orders     AS {src_db}.{src_schema}.ORDERS      PRIMARY KEY (O_ORDERKEY),
    customer   AS {src_db}.{src_schema}.CUSTOMER    PRIMARY KEY (C_CUSTKEY),
    lineitem   AS {src_db}.{src_schema}.LINEITEM    PRIMARY KEY (L_ORDERKEY, L_LINENUMBER),
    nation     AS {src_db}.{src_schema}.NATION      PRIMARY KEY (N_NATIONKEY),
    region     AS {src_db}.{src_schema}.REGION      PRIMARY KEY (R_REGIONKEY),
    part       AS {src_db}.{src_schema}.PART        PRIMARY KEY (P_PARTKEY),
    supplier   AS {src_db}.{src_schema}.SUPPLIER    PRIMARY KEY (S_SUPPKEY),
    partsupp   AS {src_db}.{src_schema}.PARTSUPP    PRIMARY KEY (PS_PARTKEY, PS_SUPPKEY)
  )
  RELATIONSHIPS (
    lineitem_to_order    AS lineitem(L_ORDERKEY)   REFERENCES orders(O_ORDERKEY),
    order_to_customer    AS orders(O_CUSTKEY)      REFERENCES customer(C_CUSTKEY),
    customer_to_nation   AS customer(C_NATIONKEY)  REFERENCES nation(N_NATIONKEY),
    supplier_to_nation   AS supplier(S_NATIONKEY)  REFERENCES nation(N_NATIONKEY),
    nation_to_region     AS nation(N_REGIONKEY)    REFERENCES region(R_REGIONKEY),
    lineitem_to_part     AS lineitem(L_PARTKEY)    REFERENCES part(P_PARTKEY),
    lineitem_to_supplier AS lineitem(L_SUPPKEY)    REFERENCES supplier(S_SUPPKEY),
    partsupp_to_part     AS partsupp(PS_PARTKEY)   REFERENCES part(P_PARTKEY),
    partsupp_to_supplier AS partsupp(PS_SUPPKEY)   REFERENCES supplier(S_SUPPKEY)
  )
  DIMENSIONS (
    orders.order_date         AS O_ORDERDATE    WITH SYNONYMS = ('date', 'order date', 'when'),
    orders.order_status       AS O_ORDERSTATUS  WITH SYNONYMS = ('status', 'state'),
    orders.order_priority     AS O_ORDERPRIORITY WITH SYNONYMS = ('priority'),
    customer.customer_name    AS C_NAME         WITH SYNONYMS = ('customer', 'client', 'buyer'),
    customer.market_segment   AS C_MKTSEGMENT   WITH SYNONYMS = ('segment', 'industry'),
    nation.nation_name        AS N_NAME         WITH SYNONYMS = ('country', 'nation'),
    region.region_name        AS R_NAME         WITH SYNONYMS = ('region', 'area', 'geography'),
    lineitem.ship_date        AS L_SHIPDATE     WITH SYNONYMS = ('ship date', 'shipped'),
    lineitem.return_flag      AS L_RETURNFLAG   WITH SYNONYMS = ('returned', 'return'),
    lineitem.ship_mode        AS L_SHIPMODE     WITH SYNONYMS = ('shipping method', 'carrier'),
    part.part_name            AS P_NAME         WITH SYNONYMS = ('part', 'item', 'product name'),
    part.brand                AS P_BRAND        WITH SYNONYMS = ('brand', 'make'),
    part.part_type            AS P_TYPE         WITH SYNONYMS = ('type', 'category', 'part type'),
    part.size                 AS P_SIZE         WITH SYNONYMS = ('size'),
    part.container            AS P_CONTAINER    WITH SYNONYMS = ('container', 'packaging'),
    supplier.supplier_name    AS S_NAME         WITH SYNONYMS = ('supplier', 'vendor'),
    supplier.supplier_phone   AS S_PHONE        WITH SYNONYMS = ('supplier phone', 'vendor phone')
  )
  METRICS (
    orders.total_order_value      AS SUM(O_TOTALPRICE)                          WITH SYNONYMS = ('revenue', 'total value', 'sales'),
    orders.order_count            AS COUNT(O_ORDERKEY)                          WITH SYNONYMS = ('number of orders', 'orders'),
    lineitem.line_revenue         AS SUM(L_EXTENDEDPRICE * (1 - L_DISCOUNT))   WITH SYNONYMS = ('net revenue', 'line total'),
    lineitem.total_quantity       AS SUM(L_QUANTITY)                            WITH SYNONYMS = ('units', 'quantity'),
    lineitem.avg_discount         AS AVG(L_DISCOUNT)                            WITH SYNONYMS = ('discount rate', 'average discount'),
    supplier.supplier_count       AS COUNT(S_SUPPKEY)                           WITH SYNONYMS = ('number of suppliers', 'suppliers'),
    supplier.total_supplier_balance AS SUM(S_ACCTBAL)                           WITH SYNONYMS = ('supplier balance', 'supplier account balance'),
    part.avg_retail_price         AS AVG(P_RETAILPRICE)                         WITH SYNONYMS = ('retail price', 'average price', 'list price'),
    partsupp.total_available_qty  AS SUM(PS_AVAILQTY)                           WITH SYNONYMS = ('available quantity', 'stock', 'inventory'),
    partsupp.avg_supply_cost      AS AVG(PS_SUPPLYCOST)                         WITH SYNONYMS = ('supply cost', 'cost', 'average cost')
  )
  COMMENT = 'TPCH SF1 semantic view (all 8 tables) for NL2SQL via Cortex Analyst'
'''.strip()

# ---- Schema + database creation, semantic view provisioning / validation for Cortex Analyst ----
provision_semantic_view = _as_bool_local(os.getenv('SNOWFLAKE_MCP_PROVISION_SEMANTIC_VIEW'), default=False)
sv_template = (os.getenv('SNOWFLAKE_MCP_SEMANTIC_VIEW_TEMPLATE') or '').strip().lower()
semantic_view_sql_env = (os.getenv('SNOWFLAKE_MCP_SEMANTIC_VIEW_SQL') or '').strip()

if provision_semantic_view:
    sv_db = (
        os.getenv('SNOWFLAKE_MCP_SEMANTIC_VIEW_DATABASE')
        or os.getenv('SNOWFLAKE_MCP_DATABASE')
        or os.getenv('SNOWFLAKE_DATABASE')
        or ''
    ).strip()
    sv_schema = (
        os.getenv('SNOWFLAKE_MCP_SEMANTIC_VIEW_SCHEMA')
        or os.getenv('SNOWFLAKE_MCP_SCHEMA')
        or os.getenv('SNOWFLAKE_SCHEMA')
        or ''
    ).strip()
    sv_name = (os.getenv('SNOWFLAKE_MCP_SEMANTIC_VIEW_NAME') or '').strip()

    sv_db = _validate_ident(sv_db, 'SNOWFLAKE_MCP_SEMANTIC_VIEW_DATABASE (or fallback database)')
    sv_schema = _validate_ident(sv_schema, 'SNOWFLAKE_MCP_SEMANTIC_VIEW_SCHEMA (or fallback schema)')
    sv_name = _validate_ident(sv_name, 'SNOWFLAKE_MCP_SEMANTIC_VIEW_NAME')

    print(f'Creating database if not exists: {sv_db}')
    _ = conn.execute_string(f'CREATE DATABASE IF NOT EXISTS {sv_db};')
    print(f'Creating schema if not exists: {sv_db}.{sv_schema}')
    _ = conn.execute_string(f'CREATE SCHEMA IF NOT EXISTS {sv_db}.{sv_schema};')

    if semantic_view_sql_env:
        semantic_view_sql = semantic_view_sql_env
        print('Provisioning semantic view from SNOWFLAKE_MCP_SEMANTIC_VIEW_SQL...')
    elif sv_template == 'tpch':
        semantic_view_sql = _build_tpch_semantic_view_sql(sv_db, sv_schema, sv_name)
        print(f'Provisioning semantic view from built-in TPCH template: {sv_db}.{sv_schema}.{sv_name}')
    else:
        raise ValueError(
            'SNOWFLAKE_MCP_PROVISION_SEMANTIC_VIEW=true requires either SNOWFLAKE_MCP_SEMANTIC_VIEW_SQL '
            'or SNOWFLAKE_MCP_SEMANTIC_VIEW_TEMPLATE=tpch.'
        )

    _ = conn.execute_string(semantic_view_sql)
    print('Semantic view provisioned.')

enable_cortex_analyst = _as_bool_local(os.getenv('SNOWFLAKE_MCP_ENABLE_CORTEX_ANALYST'), default=False)
should_resolve_semantic_identifier = provision_semantic_view or enable_cortex_analyst

if should_resolve_semantic_identifier:
    analyst_identifier = (os.getenv('SNOWFLAKE_MCP_ANALYST_IDENTIFIER') or '').strip()
    if analyst_identifier:
        target_db, target_schema, target_name = _parse_three_part_identifier(
            analyst_identifier, 'SNOWFLAKE_MCP_ANALYST_IDENTIFIER'
        )
    else:
        target_db = (os.getenv('SNOWFLAKE_MCP_SEMANTIC_VIEW_DATABASE') or os.getenv('SNOWFLAKE_MCP_DATABASE') or os.getenv('SNOWFLAKE_DATABASE') or '').strip() or None
        target_schema = (os.getenv('SNOWFLAKE_MCP_SEMANTIC_VIEW_SCHEMA') or os.getenv('SNOWFLAKE_MCP_SCHEMA') or os.getenv('SNOWFLAKE_SCHEMA') or '').strip() or None
        target_name = (os.getenv('SNOWFLAKE_MCP_SEMANTIC_VIEW_NAME') or '').strip() or None
        if target_db:
            target_db = _validate_ident(target_db, 'SNOWFLAKE_MCP_SEMANTIC_VIEW_DATABASE')
        if target_schema:
            target_schema = _validate_ident(target_schema, 'SNOWFLAKE_MCP_SEMANTIC_VIEW_SCHEMA')
        if target_name:
            target_name = _validate_ident(target_name, 'SNOWFLAKE_MCP_SEMANTIC_VIEW_NAME')

    semantic_views = _collect_semantic_views(conn, database=target_db, schema=target_schema)
    matches = _match_semantic_views(semantic_views, target_db, target_schema, target_name)

    if analyst_identifier and len(matches) != 1:
        raise ValueError(
            'SNOWFLAKE_MCP_ANALYST_IDENTIFIER not found by SHOW SEMANTIC VIEWS in target schema. '
            'Confirm the semantic view exists and the name is correct.'
        )
    if not analyst_identifier:
        if len(matches) == 0:
            raise ValueError(
                'No matching semantic view found. Set SNOWFLAKE_MCP_SEMANTIC_VIEW_NAME or '
                'SNOWFLAKE_MCP_ANALYST_IDENTIFIER, or enable provisioning with a template.'
            )
        if len(matches) > 1:
            candidates = ', '.join([f'{d}.{s}.{n}' for d, s, n in matches[:10]])
            raise ValueError('Multiple semantic views matched; set SNOWFLAKE_MCP_ANALYST_IDENTIFIER. Candidates: ' + candidates)

    resolved_db, resolved_schema, resolved_name = matches[0]
    resolved_identifier = f'{resolved_db}.{resolved_schema}.{resolved_name}'
    os.environ['SNOWFLAKE_MCP_ANALYST_IDENTIFIER'] = resolved_identifier
    print('Resolved SNOWFLAKE_MCP_ANALYST_IDENTIFIER=' + resolved_identifier)
else:
    print('Semantic view provisioning/validation skipped (Cortex Analyst not enabled and provision flag not set).')

# ---- Optional Cortex Search provisioning ----
provision_cortex_search = _as_bool_local(os.getenv('SNOWFLAKE_MCP_PROVISION_CORTEX_SEARCH'), default=False)

if not provision_cortex_search:
    print('Cortex Search provisioning skipped (SNOWFLAKE_MCP_PROVISION_CORTEX_SEARCH is not true).')
else:
    search_db = (os.getenv('SNOWFLAKE_MCP_SEARCH_DATABASE') or os.getenv('SNOWFLAKE_MCP_DATABASE') or os.getenv('SNOWFLAKE_DATABASE') or '').strip()
    search_schema = (os.getenv('SNOWFLAKE_MCP_SEARCH_SCHEMA') or os.getenv('SNOWFLAKE_MCP_SCHEMA') or os.getenv('SNOWFLAKE_SCHEMA') or '').strip()
    search_service_name = (os.getenv('SNOWFLAKE_MCP_SEARCH_SERVICE_NAME') or 'mcp_search_service').strip()
    search_warehouse = (os.getenv('SNOWFLAKE_MCP_SEARCH_WAREHOUSE') or os.getenv('SNOWFLAKE_WAREHOUSE') or '').strip()
    on_column = (os.getenv('SNOWFLAKE_MCP_SEARCH_ON_COLUMN') or '').strip()
    source_sql = (os.getenv('SNOWFLAKE_MCP_SEARCH_SOURCE_SQL') or '').strip()
    attributes_raw = (os.getenv('SNOWFLAKE_MCP_SEARCH_ATTRIBUTES') or '').strip()
    target_lag = (os.getenv('SNOWFLAKE_MCP_SEARCH_TARGET_LAG') or '1 hour').strip()

    search_db = _validate_ident(search_db, 'SNOWFLAKE_MCP_SEARCH_DATABASE')
    search_schema = _validate_ident(search_schema, 'SNOWFLAKE_MCP_SEARCH_SCHEMA')
    search_service_name = _validate_ident(search_service_name, 'SNOWFLAKE_MCP_SEARCH_SERVICE_NAME')
    search_warehouse = _validate_ident(search_warehouse, 'SNOWFLAKE_MCP_SEARCH_WAREHOUSE')
    on_column = _validate_ident(on_column, 'SNOWFLAKE_MCP_SEARCH_ON_COLUMN')
    if not source_sql:
        raise ValueError('SNOWFLAKE_MCP_SEARCH_SOURCE_SQL is required when provisioning Cortex Search.')

    attributes = [_validate_ident(x.strip(), 'SNOWFLAKE_MCP_SEARCH_ATTRIBUTES item') for x in attributes_raw.split(',') if x.strip()] if attributes_raw else []
    attr_clause = ('ATTRIBUTES ' + ', '.join(attributes)) if attributes else ''

    create_parts = [
        f'USE DATABASE {search_db};',
        f'USE SCHEMA {search_schema};',
        f'CREATE OR REPLACE CORTEX SEARCH SERVICE {search_service_name}',
        f'ON {on_column}',
    ]
    if attr_clause:
        create_parts.append(attr_clause)
    create_parts.extend([f'WAREHOUSE = {search_warehouse}', f"TARGET_LAG = '{target_lag}'", 'AS', source_sql.rstrip(';'), ';'])
    _ = conn.execute_string('\n'.join(create_parts))

    search_identifier = f'{search_db}.{search_schema}.{search_service_name}'
    os.environ['SNOWFLAKE_MCP_SEARCH_IDENTIFIER'] = search_identifier
    print('Cortex Search Service provisioned: ' + search_identifier)

## 5) Build provisioning SQL for MCP setup
Priority order:
- SNOWFLAKE_MCP_PROVISION_SQL=<inline SQL>
- If not set, a baseline CREATE MCP SERVER SQL is generated automatically.

This keeps the notebook self-contained and minimizes dependency on external SQL files.

Auto-generated SQL requires target schema settings. Prefer SNOWFLAKE_MCP_DATABASE and SNOWFLAKE_MCP_SCHEMA.
If those are not set, it falls back to SNOWFLAKE_DATABASE and SNOWFLAKE_SCHEMA.

Optional NL2SQL (Cortex Analyst) settings:
- SNOWFLAKE_MCP_ENABLE_CORTEX_ANALYST=true
- SNOWFLAKE_MCP_ANALYST_IDENTIFIER=<database.schema.semantic_view>
- SNOWFLAKE_MCP_ANALYST_TOOL_NAME=<tool name, default cortex_analyst>
- SNOWFLAKE_MCP_ANALYST_TOOL_TITLE=<title, default Cortex Analyst>
- SNOWFLAKE_MCP_ANALYST_TOOL_DESCRIPTION=<description>

Optional retrieval (Cortex Search) settings:
- SNOWFLAKE_MCP_ENABLE_CORTEX_SEARCH=true
- SNOWFLAKE_MCP_SEARCH_IDENTIFIER=<database.schema.cortex_search_service>
- SNOWFLAKE_MCP_SEARCH_TOOL_NAME=<tool name, default cortex_search>
- SNOWFLAKE_MCP_SEARCH_TOOL_TITLE=<title, default Cortex Search>
- SNOWFLAKE_MCP_SEARCH_TOOL_DESCRIPTION=<description>

Note: Step 4 can resolve and set identifiers for Analyst and Search in this notebook session.

In [ ]:
def _validate_simple_identifier(value: str, env_name: str) -> str:
    import re
    if not value:
        raise ValueError(f'{env_name} is required and cannot be empty.')
    if not re.match(r'^[A-Za-z_][A-Za-z0-9_$]*$', value):
        raise ValueError(
            f'{env_name} has unsupported characters: {value}. Use a simple unquoted identifier.'
        )
    return value

def _as_bool(value: str | None, default: bool = False) -> bool:
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

def _build_default_mcp_sql() -> str:
    database = (os.getenv('SNOWFLAKE_MCP_DATABASE') or os.getenv('SNOWFLAKE_DATABASE') or '').strip()
    schema = (os.getenv('SNOWFLAKE_MCP_SCHEMA') or os.getenv('SNOWFLAKE_SCHEMA') or '').strip()
    server_name = os.getenv('SNOWFLAKE_MCP_SERVER_NAME', 'TRIAL_MCP_SERVER').strip()

    enable_sql_exec_tool = _as_bool(os.getenv('SNOWFLAKE_MCP_ENABLE_SQL_EXEC_TOOL'), default=True)
    sql_tool_name = os.getenv('SNOWFLAKE_MCP_TOOL_NAME', 'sql_exec_tool').strip()
    sql_tool_title = os.getenv('SNOWFLAKE_MCP_TOOL_TITLE', 'SQL Execution Tool').strip()
    sql_tool_description = os.getenv(
        'SNOWFLAKE_MCP_TOOL_DESCRIPTION',
        'A tool to execute SQL queries against the connected Snowflake database.'
    ).strip()

    enable_cortex_analyst = _as_bool(os.getenv('SNOWFLAKE_MCP_ENABLE_CORTEX_ANALYST'), default=False)
    analyst_identifier = os.getenv('SNOWFLAKE_MCP_ANALYST_IDENTIFIER', '').strip()
    analyst_tool_name = os.getenv('SNOWFLAKE_MCP_ANALYST_TOOL_NAME', 'cortex_analyst').strip()
    analyst_tool_title = os.getenv('SNOWFLAKE_MCP_ANALYST_TOOL_TITLE', 'Cortex Analyst').strip()
    analyst_tool_description = os.getenv(
        'SNOWFLAKE_MCP_ANALYST_TOOL_DESCRIPTION',
        'Natural language to SQL via Cortex Analyst semantic view.'
    ).strip()

    enable_cortex_search = _as_bool(os.getenv('SNOWFLAKE_MCP_ENABLE_CORTEX_SEARCH'), default=False)
    search_identifier = os.getenv('SNOWFLAKE_MCP_SEARCH_IDENTIFIER', '').strip()
    search_tool_name = os.getenv('SNOWFLAKE_MCP_SEARCH_TOOL_NAME', 'cortex_search').strip()
    search_tool_title = os.getenv('SNOWFLAKE_MCP_SEARCH_TOOL_TITLE', 'Cortex Search').strip()
    search_tool_description = os.getenv(
        'SNOWFLAKE_MCP_SEARCH_TOOL_DESCRIPTION',
        'Retrieval over a Cortex Search Service for grounded responses.'
    ).strip()

    database = _validate_simple_identifier(database, 'SNOWFLAKE_MCP_DATABASE or SNOWFLAKE_DATABASE')
    schema = _validate_simple_identifier(schema, 'SNOWFLAKE_MCP_SCHEMA or SNOWFLAKE_SCHEMA')
    server_name = _validate_simple_identifier(server_name, 'SNOWFLAKE_MCP_SERVER_NAME')

    if enable_sql_exec_tool:
        sql_tool_name = _validate_simple_identifier(sql_tool_name, 'SNOWFLAKE_MCP_TOOL_NAME')

    if enable_cortex_analyst:
        analyst_tool_name = _validate_simple_identifier(analyst_tool_name, 'SNOWFLAKE_MCP_ANALYST_TOOL_NAME')
        if not analyst_identifier:
            raise ValueError(
                'SNOWFLAKE_MCP_ANALYST_IDENTIFIER is required when SNOWFLAKE_MCP_ENABLE_CORTEX_ANALYST=true.'
            )

    if enable_cortex_search:
        search_tool_name = _validate_simple_identifier(search_tool_name, 'SNOWFLAKE_MCP_SEARCH_TOOL_NAME')
        if not search_identifier:
            raise ValueError(
                'SNOWFLAKE_MCP_SEARCH_IDENTIFIER is required when SNOWFLAKE_MCP_ENABLE_CORTEX_SEARCH=true.'
            )

    def _yaml_quote(text: str) -> str:
        return text.replace("'", "''")

    tools_yaml = []

    if enable_sql_exec_tool:
        tools_yaml.append(
            "\n".join(
                [
                    f"      - title: '{_yaml_quote(sql_tool_title)}'",
                    f"        name: '{sql_tool_name}'",
                    "        type: 'SYSTEM_EXECUTE_SQL'",
                    f"        description: '{_yaml_quote(sql_tool_description)}'",
                ]
            )
        )

    if enable_cortex_analyst:
        tools_yaml.append(
            "\n".join(
                [
                    f"      - title: '{_yaml_quote(analyst_tool_title)}'",
                    f"        name: '{analyst_tool_name}'",
                    "        type: 'CORTEX_ANALYST_MESSAGE'",
                    f"        identifier: '{_yaml_quote(analyst_identifier)}'",
                    f"        description: '{_yaml_quote(analyst_tool_description)}'",
                ]
            )
        )

    if enable_cortex_search:
        tools_yaml.append(
            "\n".join(
                [
                    f"      - title: '{_yaml_quote(search_tool_title)}'",
                    f"        name: '{search_tool_name}'",
                    "        type: 'CORTEX_SEARCH_SERVICE_QUERY'",
                    f"        identifier: '{_yaml_quote(search_identifier)}'",
                    f"        description: '{_yaml_quote(search_tool_description)}'",
                ]
            )
        )

    if not tools_yaml:
        raise ValueError(
            'No MCP tools configured. Enable SQL_EXEC, Cortex Analyst, or Cortex Search tool in environment variables.'
        )

    tools_block = "\n".join(tools_yaml)

    return f'''
USE DATABASE {database};
USE SCHEMA {schema};
CREATE MCP SERVER IF NOT EXISTS {server_name}
  FROM SPECIFICATION $$
    tools:
{tools_block}
  $$;
SHOW MCP SERVERS LIKE '{server_name}' IN SCHEMA {database}.{schema};
DESCRIBE MCP SERVER {server_name};
'''.strip()

sql_inline_value = os.getenv('SNOWFLAKE_MCP_PROVISION_SQL')

if sql_inline_value:
    provisioning_sql = sql_inline_value
    sql_source = 'inline env var SNOWFLAKE_MCP_PROVISION_SQL'
else:
    provisioning_sql = _build_default_mcp_sql()
    sql_source = 'auto-generated baseline MCP server SQL'

if not provisioning_sql.strip():
    raise ValueError('Provisioning SQL is empty.')

print('Provisioning SQL source:', sql_source)
print('SQL length (chars):', len(provisioning_sql))
print('Target DB/Schema:', (os.getenv('SNOWFLAKE_MCP_DATABASE') or os.getenv('SNOWFLAKE_DATABASE')), '/', (os.getenv('SNOWFLAKE_MCP_SCHEMA') or os.getenv('SNOWFLAKE_SCHEMA')))
print('Cortex Analyst enabled:', _as_bool(os.getenv('SNOWFLAKE_MCP_ENABLE_CORTEX_ANALYST'), default=False))
print('Cortex Search enabled:', _as_bool(os.getenv('SNOWFLAKE_MCP_ENABLE_CORTEX_SEARCH'), default=False))
print('SQL preview:')
print(provisioning_sql)

## 6) Execute provisioning SQL
This cell executes the SQL built in the previous build step.

Tip: edit the build step first if you want to customize what this step runs.

In [ ]:
results = conn.execute_string(provisioning_sql)
statement_count = 0
for c in results:
    statement_count += 1
    try:
        rows = c.fetchall()
        print(f'Statement {statement_count}: returned {len(rows)} row(s)')
    except Exception:
        print(f'Statement {statement_count}: executed')
print(f'Provisioning complete. Statements executed: {statement_count}')

## 7) Optional validation SQL
If you set SNOWFLAKE_MCP_VALIDATE_SQL in .env, this cell runs it and prints the result.

In [ ]:
validate_sql = os.getenv('SNOWFLAKE_MCP_VALIDATE_SQL', '').strip()
if not validate_sql:
    database = (os.getenv('SNOWFLAKE_MCP_DATABASE') or os.getenv('SNOWFLAKE_DATABASE') or '').strip()
    schema = (os.getenv('SNOWFLAKE_MCP_SCHEMA') or os.getenv('SNOWFLAKE_SCHEMA') or '').strip()
    server_name = os.getenv('SNOWFLAKE_MCP_SERVER_NAME', 'TRIAL_MCP_SERVER').strip()
    if database and schema:
        validate_sql = f"SHOW MCP SERVERS LIKE '{server_name}' IN SCHEMA {database}.{schema};"
        print('No SNOWFLAKE_MCP_VALIDATE_SQL set; using default validation query.')
    else:
        print('No SNOWFLAKE_MCP_VALIDATE_SQL set and database/schema are missing; skipping validation query.')

if not validate_sql:
    pass
else:
    with conn.cursor() as cur:
        cur.execute(validate_sql)
        data = cur.fetchall()
        print(f'Validation rows: {len(data)}')
        for row in data:
            print(row)

## Copilot Studio MCP Configuration
Run the next cell to print ready-to-copy values for the Copilot Studio MCP onboarding wizard.

**Prerequisites:**
- Run `snowflake_oauth_lab_setup.ipynb` first -- it creates the Entra app registrations
  and writes `SNOWFLAKE_CLIENT_ID`, `SNOWFLAKE_CLIENT_SECRET`, `AZURE_TENANT_ID`, and
  `SNOWFLAKE_APP_ID_URI` to `.env`.
- In Copilot Studio, enable **Generative orchestration** on your agent before adding
  the MCP server. MCP tools are only available when generative orchestration is on.
- Use **OAuth 2.0 -> Manual** in the Copilot Studio MCP wizard (Snowflake does not
  support dynamic client registration).

In [ ]:
mcp_database = (os.getenv('SNOWFLAKE_MCP_DATABASE') or os.getenv('SNOWFLAKE_DATABASE') or '').strip()
mcp_schema = (os.getenv('SNOWFLAKE_MCP_SCHEMA') or os.getenv('SNOWFLAKE_SCHEMA') or '').strip()
mcp_server_name = (os.getenv('SNOWFLAKE_MCP_SERVER_NAME') or 'TRIAL_MCP_SERVER').strip()

host = ''
if 'conn' in globals() and getattr(conn, 'host', None):
    host = str(conn.host).strip()
if not host:
    account = (os.getenv('SNOWFLAKE_ACCOUNT') or '').strip()
    if account:
        host = account + '.snowflakecomputing.com'

host = host.replace('https://', '').replace('http://', '').strip('/')
# Snowflake docs: use hyphens instead of underscores in hostnames for MCP connections.
# Underscores in the hostname cause tool discovery failures.
host = host.replace('_', '-')

mcp_server_url = (
    'https://'
    + host
    + '/api/v2/databases/' + mcp_database
    + '/schemas/' + mcp_schema
    + '/mcp-servers/' + mcp_server_name
)

tenant_id = (os.getenv('AZURE_TENANT_ID') or os.getenv('TENANT_ID') or '').strip()
app_id_uri = (os.getenv('SNOWFLAKE_APP_ID_URI') or '').strip()
client_id = (os.getenv('SNOWFLAKE_CLIENT_ID') or '').strip()
client_secret_set = bool((os.getenv('SNOWFLAKE_CLIENT_SECRET') or '').strip())

authorization_url = ''
token_url_template = ''
refresh_url = ''
oauth_scope = ''

if tenant_id:
    authorization_url = f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/authorize'
    token_url_template = f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token'
    refresh_url = token_url_template

if app_id_uri:
    oauth_scope = app_id_uri + '/.default'

warnings = []
if not client_id:
    warnings.append('SNOWFLAKE_CLIENT_ID is not set -- run snowflake_oauth_lab_setup.ipynb first')
if not client_secret_set:
    warnings.append('SNOWFLAKE_CLIENT_SECRET is not set -- run snowflake_oauth_lab_setup.ipynb first')
if not tenant_id:
    warnings.append('AZURE_TENANT_ID / TENANT_ID is not set')
if not app_id_uri:
    warnings.append('SNOWFLAKE_APP_ID_URI is not set')

if warnings:
    print('WARNING - missing values:')
    for w in warnings:
        print('  * ' + w)
    print('')

print('Copilot Studio MCP onboarding wizard values:')
print('')
print('  Server name:        ' + mcp_server_name)
print('  Server description: Snowflake MCP server (NL2SQL, search, SQL execution)')
print('  Server URL:         ' + mcp_server_url)
print('')
print('  Authentication:     OAuth 2.0')
print('  OAuth type:         Manual   <-- choose this; Snowflake does not support dynamic client registration')
print('')
print('  Client ID:          ' + (client_id or '(not set -- run snowflake_oauth_lab_setup.ipynb)'))
print('  Client secret:      ' + ('[set]' if client_secret_set else '(not set -- run snowflake_oauth_lab_setup.ipynb)'))
print('  Authorization URL:  ' + (authorization_url or '(set AZURE_TENANT_ID)'))
print('  Token URL:          ' + (token_url_template or '(set AZURE_TENANT_ID)'))
print('  Refresh URL:        ' + (refresh_url or '(set AZURE_TENANT_ID)'))
print('  Scopes:             ' + (oauth_scope or '(set SNOWFLAKE_APP_ID_URI)'))
print('')
print('  After saving in Copilot Studio:')
print('    1. Copy the callback URL that Copilot Studio shows.')
print('    2. Add it as a Redirect URI in the Entra client app registration (SNOWFLAKE_CLIENT_APP_NAME).')
print('    3. Enable Generative orchestration on the agent if not already on.')
print('    4. Re-open the MCP server connection -- tools will appear once OAuth completes.')


## VS Code MCP Configuration
Use this cell to print a ready-to-paste VS Code `mcp.json` configuration.

Snowflake MCP uses Streamable HTTP transport (MCP spec 2025-11-25). Use `type: "http"` in VS Code,
not `type: "sse"`. There is no `/sse` path -- tools/list is a POST to the base URL.

Authentication uses a Programmatic Access Token (PAT). Create one in Snowflake:
Admin -> Users & Roles -> select user -> Programmatic Access Tokens -> Generate.

In [ ]:
import json as _json

# mcp_server_url must be set by running the Copilot Studio config cell above first.
_url = mcp_server_url if 'mcp_server_url' in dir() else 'run Copilot Studio cell above first'

vscode_mcp_config = {
    'servers': {
        mcp_server_name: {
            'type': 'http',
            'url': _url,
            'headers': {
                'Authorization': 'Bearer YOUR-PAT-TOKEN'
            }
        }
    }
}

print('.vscode/mcp.json (Streamable HTTP, PAT auth):')
print(_json.dumps(vscode_mcp_config, indent=2))
print('')
print('Notes:')
print('  1. Replace YOUR-PAT-TOKEN with your Snowflake Programmatic Access Token.')
print('  2. Use type=http (Streamable HTTP). Snowflake MCP has no /sse path -- SSE is not supported.')
print('  3. Do not commit mcp.json to source control if it contains your PAT token.')
print('  4. The role used for the PAT must have USAGE on the MCP server and its tools.')


## 8) OAuth User Access Check (from OAuth lab assumptions)
This step verifies the OAuth user and role grants needed for table access and MCP usage.

Set `SNOWFLAKE_MCP_APPLY_OAUTH_GRANTS=true` in `.env` to auto-apply any missing grants.
By default, this cell only reports what is missing and prints the SQL to run.

In [ ]:
import os
from snowflake.connector import DictCursor

def _as_bool_oauth(value: str | None, default: bool = False) -> bool:
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

oauth_user = (os.getenv('SNOWFLAKE_OAUTH_USER') or 'SNOWSQL_OAUTH_USER').strip()
oauth_role = (os.getenv('SNOWFLAKE_OAUTH_ROLE') or 'ANALYST').strip()
warehouse = (os.getenv('SNOWFLAKE_WAREHOUSE') or '').strip()

sample_db = (os.getenv('SNOWFLAKE_DATABASE') or 'SNOWFLAKE_SAMPLE_DATA').strip()
sample_schema = (os.getenv('SNOWFLAKE_SCHEMA') or 'TPCH_SF1').strip()

mcp_db = (os.getenv('SNOWFLAKE_MCP_DATABASE') or os.getenv('SNOWFLAKE_DATABASE') or '').strip()
mcp_schema = (os.getenv('SNOWFLAKE_MCP_SCHEMA') or os.getenv('SNOWFLAKE_SCHEMA') or '').strip()
mcp_sv = (os.getenv('SNOWFLAKE_MCP_SEMANTIC_VIEW_NAME') or 'TPCH_NL2SQL').strip()
mcp_server = (os.getenv('SNOWFLAKE_MCP_SERVER_NAME') or '').strip()

apply_missing = _as_bool_oauth(os.getenv('SNOWFLAKE_MCP_APPLY_OAUTH_GRANTS'), default=False)

print('OAuth access target:')
print('  user:', oauth_user)
print('  role:', oauth_role)
print('  sample target:', f'{sample_db}.{sample_schema}')
print('  mcp target:', f'{mcp_db}.{mcp_schema}.{mcp_sv}')
print('  mcp server:', mcp_server)
print('  apply missing grants:', apply_missing)

with conn.cursor(DictCursor) as cur:
    cur.execute(f"SHOW USERS LIKE '{oauth_user}'")
    user_rows = cur.fetchall()

if not user_rows:
    raise ValueError(f'OAuth user not found: {oauth_user}')

with conn.cursor(DictCursor) as cur:
    cur.execute(f"SHOW GRANTS TO USER {oauth_user}")
    grants_to_user = cur.fetchall()

user_has_role = any(
    str((r.get('name') or '')).upper() == oauth_role.upper() and str((r.get('granted_on') or '')).upper() == 'ROLE'
    for r in grants_to_user
)

with conn.cursor(DictCursor) as cur:
    cur.execute(f"SHOW GRANTS TO ROLE {oauth_role}")
    grants_to_role = cur.fetchall()

role_grants = [
    (
        str((r.get('privilege') or '')).upper(),
        str((r.get('granted_on') or '')).upper(),
        str((r.get('name') or '')).upper(),
    )
    for r in grants_to_role
]

def _has_role_grant(privilege: str, granted_on: str, object_name: str) -> bool:
    p = privilege.upper()
    go = granted_on.upper()
    n = object_name.upper()
    return any(rp == p and rgo == go and rn == n for rp, rgo, rn in role_grants)

missing_sql = []

if not user_has_role:
    missing_sql.append(f'GRANT ROLE {oauth_role} TO USER {oauth_user};')

if warehouse and not _has_role_grant('USAGE', 'WAREHOUSE', warehouse):
    missing_sql.append(f'GRANT USAGE ON WAREHOUSE {warehouse} TO ROLE {oauth_role};')

if sample_db.upper() == 'SNOWFLAKE_SAMPLE_DATA':
    has_sample_usage = _has_role_grant('USAGE', 'DATABASE', sample_db)
    if not has_sample_usage:
        missing_sql.append(f'GRANT IMPORTED PRIVILEGES ON DATABASE {sample_db} TO ROLE {oauth_role};')
else:
    if not _has_role_grant('USAGE', 'DATABASE', sample_db):
        missing_sql.append(f'GRANT USAGE ON DATABASE {sample_db} TO ROLE {oauth_role};')
    if not _has_role_grant('USAGE', 'SCHEMA', f'{sample_db}.{sample_schema}'):
        missing_sql.append(f'GRANT USAGE ON SCHEMA {sample_db}.{sample_schema} TO ROLE {oauth_role};')
    missing_sql.append(f'GRANT SELECT ON ALL TABLES IN SCHEMA {sample_db}.{sample_schema} TO ROLE {oauth_role};')
    missing_sql.append(f'GRANT SELECT ON FUTURE TABLES IN SCHEMA {sample_db}.{sample_schema} TO ROLE {oauth_role};')

if mcp_db and not _has_role_grant('USAGE', 'DATABASE', mcp_db):
    missing_sql.append(f'GRANT USAGE ON DATABASE {mcp_db} TO ROLE {oauth_role};')
if mcp_db and mcp_schema and not _has_role_grant('USAGE', 'SCHEMA', f'{mcp_db}.{mcp_schema}'):
    missing_sql.append(f'GRANT USAGE ON SCHEMA {mcp_db}.{mcp_schema} TO ROLE {oauth_role};')

with conn.cursor(DictCursor) as cur:
    cur.execute(f"SHOW GRANTS ON SEMANTIC VIEW {mcp_db}.{mcp_schema}.{mcp_sv}")
    sv_grants = cur.fetchall()

sv_access = any(
    str((r.get('grantee_name') or '')).upper() == oauth_role.upper() and str((r.get('privilege') or '')).upper() in {'SELECT', 'OWNERSHIP'}
    for r in sv_grants
)
if not sv_access:
    missing_sql.append(f'GRANT SELECT ON SEMANTIC VIEW {mcp_db}.{mcp_schema}.{mcp_sv} TO ROLE {oauth_role};')

with conn.cursor(DictCursor) as cur:
    cur.execute(f"USE DATABASE {mcp_db}")
    cur.execute(f"USE SCHEMA {mcp_schema}")
    cur.execute(f"SHOW GRANTS ON MCP SERVER {mcp_server}")
    mcp_grants = cur.fetchall()

mcp_access = any(
    str((r.get('grantee_name') or '')).upper() == oauth_role.upper() and str((r.get('privilege') or '')).upper() in {'USAGE', 'OWNERSHIP'}
    for r in mcp_grants
)
if not mcp_access:
    missing_sql.append(f'GRANT USAGE ON MCP SERVER {mcp_server} TO ROLE {oauth_role};')

print('')
print('Access check results:')
print('  user exists:', bool(user_rows))
print('  user has role grant:', user_has_role)
print('  semantic view access:', sv_access)
print('  mcp server access:', mcp_access)
print('  missing statements:', len(missing_sql))

if missing_sql:
    print('')
    print('Required SQL to align with OAuth lab access assumptions:')
    for stmt in missing_sql:
        print(stmt)

if apply_missing and missing_sql:
    print('')
    print('Applying missing grants...')
    with conn.cursor() as cur:
        for stmt in missing_sql:
            cur.execute(stmt)
            print('Applied:', stmt)
    print('Grant application complete.')
elif apply_missing and not missing_sql:
    print('')
    print('No missing grants to apply.')

## 9) Cleanup connection

In [ ]:
try:
    conn.close()
    print('Snowflake connection closed.')
except NameError:
    print('No active connection found.')